In [1]:
!pip install yfinance psycopg2-binary pandas

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 2.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 12.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 14.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.3/143.3 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.6/184.6 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.7/153.7 kB 9.2 MB/s eta 0:00:00
  Created wheel for multitasking: filename=multitasking-0.0.12-py3-none-any.whl size=15548 sha256=ed3f71732b374cc95d21741e54b7c27b83a69b2ae83a766b0554af21771da52f
  Stored in directory: /home/jovyan/.cache/pip/wheels/42/d6/84/bf57a755f4569494cd00de4bb46ef064874823f4d19c82e960
Successfully built multitasking
  Attempting uninstall: certifi
    Found existing installation: certifi 2023.7.22
    Uninstalling certifi-2023.7.22:
      Successfully uninstal

In [2]:
import yfinance as yf
import pandas as pd
import psycopg2

In [3]:
symbol = "AAPL"
df = yf.download(
    symbol,
    start="2023-01-01",
    end="2025-01-01",
    interval="1d"
)

df = df.reset_index()
df.head()

[*********************100%***********************]  1 of 1 completed


Price,Date,Close,High,Low,Open,Volume
Ticker,,AAPL,AAPL,AAPL,AAPL,AAPL
0,2023-01-03,123.096024,128.834003,122.210227,128.223793,112117500
1,2023-01-04,124.365669,126.629372,123.105873,124.887303,89113600
2,2023-01-05,123.046806,125.753403,122.790915,125.123505,80962700
3,2023-01-06,127.574219,128.233642,122.918876,124.021202,87754700
4,2023-01-09,128.095840,131.304398,127.839949,128.410797,70790800


In [4]:
import pandas as pd

if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

df = df.reset_index()

In [5]:
print(type(df.loc[0, "Volume"]))

<class 'numpy.int64'>


In [6]:
#transform
df["return"] = df["Close"].pct_change()

# rolling volatility 20 hari
df["volatility_20d"] = df["return"].rolling(20).std()

df = df.dropna()
df.head()

Price,index,Date,Close,High,Low,Open,Volume,return,volatility_20d
20,20,2023-02-01,143.134674,144.296058,139.089557,141.697726,77663600,0.007901,0.012758
21,21,2023-02-02,148.439606,148.793909,145.831422,146.549896,118339000,0.037063,0.014354
22,22,2023-02-03,152.061554,154.896104,145.496827,145.693668,154357300,0.024400,0.013969
23,23,2023-02-06,149.335251,150.683638,148.400248,150.162005,69858300,-0.017929,0.013955
24,24,2023-02-07,152.209152,152.780000,148.262447,148.262447,83322600,0.019245,0.014142


In [7]:
df = df.reset_index(drop=True)

df["Volume"] = df["Volume"].astype(int)
df["return"] = df["return"].astype(float)
df["volatility_20d"] = df["volatility_20d"].astype(float)

In [8]:
conn = psycopg2.connect(
    host="postgres",
    port=5432,
    database="batch_db",
    user="airflow",
    password="airflow"
)
cur = conn.cursor()

In [9]:
for _, row in df.iterrows():
    cur.execute("""
        INSERT INTO yahoo_finance.stock_price
        (symbol, date, open, high, low, close, volume, return, volatility_20d)
        VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)
    """, (
        symbol,
        row["Date"],
        row["Open"],
        row["High"],
        row["Low"],
        row["Close"],
        int(row["Volume"]),
        float(row["return"]),
        float(row["volatility_20d"])
    ))

conn.commit()
cur.close()
conn.close()

print("✅ Historical data loaded to Postgres")

✅ Historical data loaded to Postgres
